# Run SDK-Backed Data Agent Evaluation

This notebook runs the real, Microsoft-recommended evaluation path (`fabric-data-agent-sdk`) against a deployed Fabric Data Agent, so you do not have to type challenge questions into the chat UI by hand.

It downloads `evaluation/evaluate_agent.py` and a question dataset from this repository, then calls `DataAgentEvaluator.evaluate_with_sdk(...)` directly. Results are printed inline and saved to a JSON file (plus an optional CSV of the official SDK detail rows) in this notebook's local storage.

This notebook only supports `--sdk-mode` (real evaluation). It never simulates answers.

## Required Fabric setup

Before running the notebook:

1. In the notebook **Explorer** pane, select **Add data items** and attach the Lakehouse that contains the SDK evaluation output table.
2. Make that Lakehouse the notebook's **default Lakehouse**.
3. Expand the Lakehouse's **Tables** folder, copy the exact existing table name, and set `TABLE_NAME` in section 2 to that value. Do not use a file name or a table from a different Lakehouse.

If Fabric reports `Table does not exist`, correct the attachment/default Lakehouse or `TABLE_NAME`, then rerun from section 2.

In [ ]:
# 1. Install compatible SDK dependencies, then restart the Fabric Python interpreter.

import subprocess

import sys



packages = [

    "fabric-data-agent-sdk>=0.1.30a0",

    "pandas",

    "typing_extensions>=4.12.2",

    "PyJWT>=2.6.0",

]

subprocess.check_call(

    [sys.executable, "-m", "pip", "install", "-q", "-U", *packages]

)



verification = subprocess.run(

    [

        sys.executable,

        "-c",

        "from fabric.dataagent.evaluation import "

        "evaluate_data_agent, get_evaluation_details, get_evaluation_summary; "

        "from typing_extensions import Sentinel; "

        "print('Fabric Data Agent evaluation SDK dependencies verified')",

    ],

    capture_output=True,

    text=True,

)

if verification.returncode != 0:

    raise RuntimeError(

        "The packages installed, but a fresh Python process could not import them.\n"

        f"{verification.stderr.strip()}"

    )

print(verification.stdout.strip())

print("Restarting the Fabric Python interpreter; execution will continue in the next cell.")

notebookutils.session.restartPython()


## 2. Configure and verify the restarted kernel

In [ ]:
import importlib



fabric_evaluation = importlib.import_module("fabric.dataagent.evaluation")

from typing_extensions import Sentinel



print("Current notebook kernel import verified:", fabric_evaluation.__name__)



# Evaluator parameters

WORKSPACE_NAME = "Hackathon"

AGENT_NAME = "LegalFirmAgent"

DATASET_NAME = "challenge"  # "challenge" (6 questions) or "routing" (Step 5 extension)

SNAPSHOT_NAME = "final"  # use "baseline" before tuning and "final" after tuning

INCLUDE_PARAPHRASES = True  # challenge becomes 12 SDK prompts

# Must exactly match an existing table under the notebook's attached default Lakehouse.
TABLE_NAME = f"demo_evaluation_{SNAPSHOT_NAME}"

DATA_AGENT_STAGE = "sandbox"  # "sandbox"/"draft" for unpublished agents; "production" after publish

CRITIC_PROMPT = ""  # optional stricter/domain-specific evaluator prompt

REPOSITORY_OWNER = "Limaoncloud"

REPOSITORY_NAME = "fabric-data-agent-hackathon"

REPOSITORY_REF = "dev"

OUTPUT_PATH = f"{SNAPSHOT_NAME}_sdk_evaluation_results.json"

SAVE_OFFICIAL_DETAILS_CSV = True


## 3. Download the evaluator module and question dataset



Downloads directly from this repository at `REPOSITORY_REF`, mirroring how `NB_Deploy_Data_Agent_Hackathon.ipynb` downloads its deployer module and profile.

In [ ]:
import json

import tempfile

import types

from pathlib import Path



import requests



raw_base_url = (

    f"https://raw.githubusercontent.com/{REPOSITORY_OWNER}/"

    f"{REPOSITORY_NAME}/{REPOSITORY_REF}"

)

evaluator_url = f"{raw_base_url}/evaluation/evaluate_agent.py"

dataset_url = f"{raw_base_url}/evaluation/{DATASET_NAME}/uk-legal.json"



evaluator_response = requests.get(evaluator_url, timeout=180)

evaluator_response.raise_for_status()

evaluate_agent = types.ModuleType("evaluate_agent")

exec(compile(evaluator_response.text, evaluator_url, "exec"), evaluate_agent.__dict__)



dataset_response = requests.get(dataset_url, timeout=180)

dataset_response.raise_for_status()

dataset = dataset_response.json()



if INCLUDE_PARAPHRASES:

    expanded_queries = []

    for item in dataset["evaluation_queries"]:

        original = {**item, "original_id": item["id"], "variant": "original"}

        paraphrase = {

            **item,

            "id": f"{item['id']}-P",

            "original_id": item["id"],

            "question": item["paraphrase"],

            "variant": "paraphrase",

        }

        expanded_queries.extend([original, paraphrase])

    dataset["evaluation_queries"] = expanded_queries

    dataset["metadata"]["total_queries"] = len(expanded_queries)



dataset_path = Path(tempfile.gettempdir()) / f"{SNAPSHOT_NAME}_{DATASET_NAME}_uk-legal.json"

dataset_path.write_text(json.dumps(dataset, indent=2), encoding="utf-8")



print("Evaluator module:", evaluator_url)

print("Dataset:", dataset_url)

print("Snapshot:", SNAPSHOT_NAME)

print("SDK prompts:", len(dataset["evaluation_queries"]))

print("Dataset saved to:", dataset_path)


## 4. Run the real SDK-backed snapshot



Run once with `SNAPSHOT_NAME = "baseline"` before tuning and again with `SNAPSHOT_NAME = "final"` after tuning. With `INCLUDE_PARAPHRASES = True`, the SDK evaluates all 12 challenge prompts.

In [ ]:
valid_data_agent_stages = {"sandbox", "draft", "production"}

data_agent_stage = DATA_AGENT_STAGE.strip().lower()

if data_agent_stage not in valid_data_agent_stages:

    raise ValueError(

        f"DATA_AGENT_STAGE must be one of {sorted(valid_data_agent_stages)}; "

        f"received {DATA_AGENT_STAGE!r}."

    )



evaluator = evaluate_agent.DataAgentEvaluator(

    agent_id=AGENT_NAME,

    sdk_mode=True,

    workspace_name=WORKSPACE_NAME or None,

    table_name=TABLE_NAME,

    data_agent_stage=data_agent_stage,

    critic_prompt=CRITIC_PROMPT or None,

)



metrics_list, aggregate, sdk_context = evaluator.evaluate_with_sdk(str(dataset_path))

evaluator.print_report(aggregate)

evaluator.print_compatibility_report(aggregate, sdk_context)


## 5. Save the snapshot and evidence



The JSON includes mapped answers, selected sources, SQL/DAX when exposed by the SDK, and scoring diagnostics. The CSV preserves every raw official detail column for audit evidence.

In [ ]:
evaluator.save_results(metrics_list, aggregate, OUTPUT_PATH, sdk_context)

if SAVE_OFFICIAL_DETAILS_CSV:
    csv_path = OUTPUT_PATH.rsplit(".", 1)[0] + "_official_details.csv"
    evaluator.save_official_details_csv(csv_path)
